# FinSurvival Competition: Starter Notebook (Cox Model Prediction Submission)

**Objective:** This notebook provides a workflow for creating a valid prediction submission using the `CoxPHFitter` model. The competition requires you to submit a `.zip` file containing 16 separate prediction files in CSV format.

This notebook will guide you through:
1.  Loading the training and test sets for each of the 16 tasks from a single directory.
2.  Training a model (using `CoxPHFitter` as an example).
3.  Generating predictions on the test set in the required format.
4.  Saving each set of predictions to a correctly named CSV file.
5.  Zipping all 16 prediction files for submission.

## Step 1: Setup and Imports

In [1]:
# Install required packages
# pip install -q pandas lifelines==0.27.8 scikit-learn==1.2.2 scikit-survival==0.21.0 numpy

# Import libraries
import pandas as pd
import numpy as np
import os
import shutil
from lifelines import CoxPHFitter
from lifelines.exceptions import ConvergenceError
from sklearn.preprocessing import StandardScaler
from typing import Tuple, Optional

## Step 2: Define a Preprocessing Function

Even though you are not submitting this code, you will still need a preprocessing pipeline to train your models effectively. You can use the one below as a starting point.

In [2]:
def preprocess(
    train_df_with_labels: pd.DataFrame,
    test_features_df: Optional[pd.DataFrame] = None,
) -> Tuple[pd.DataFrame, pd.DataFrame, Optional[pd.DataFrame]]:
    """
    Preprocesses data for the competition.
    """
    train_targets = train_df_with_labels[["timeDiff", "status"]]
    train_features = train_df_with_labels.drop(columns=["timeDiff", "status"])
    cols_to_drop = ["id", "user", "pool", "Index Event", "Outcome Event", "type", "timestamp"]
    train_features = train_features.drop(columns=cols_to_drop, errors="ignore")
    categorical_cols = train_features.select_dtypes(include=["object", "category"]).columns
    for col in categorical_cols:
        top_categories = train_features[col].value_counts().nlargest(10).index
        train_features[col] = train_features[col].where(train_features[col].isin(top_categories), "Other")
    train_features_encoded = pd.get_dummies(train_features, columns=categorical_cols, dummy_na=True, drop_first=True)
    numerical_cols = train_features_encoded.select_dtypes(include=np.number).columns
    scaler = StandardScaler()
    train_features_scaled = scaler.fit_transform(train_features_encoded[numerical_cols])
    train_features_final = pd.DataFrame(train_features_scaled, index=train_features_encoded.index, columns=numerical_cols).fillna(0)
    cols_to_keep = train_features_final.columns[train_features_final.var() != 0]
    train_features_final = train_features_final[cols_to_keep]
    test_processed_features = None
    if test_features_df is not None:
        test_features = test_features_df.drop(columns=cols_to_drop, errors="ignore")
        for col in categorical_cols:
            top_categories = train_features[col].value_counts().nlargest(10).index
            test_features[col] = test_features[col].where(test_features[col].isin(top_categories), "Other")
        test_features_encoded = pd.get_dummies(test_features, columns=categorical_cols, dummy_na=True, drop_first=True)
        train_cols = train_features_encoded.columns
        test_features_aligned = test_features_encoded.reindex(columns=train_cols, fill_value=0)
        test_features_scaled = scaler.transform(test_features_aligned[numerical_cols])
        test_features_final = pd.DataFrame(test_features_scaled, index=test_features_aligned.index, columns=numerical_cols).fillna(0)
        test_processed_features = test_features_final[cols_to_keep]
    return train_features_final, train_targets, test_processed_features

## Step 3: Loop, Train, and Save Predictions

This is the main part of the notebook. We will loop through all 16 tasks. For each task, we will:
1. Load the training data and the test features.
2. Preprocess both.
3. Train a model on the training data.
4. Generate predictions on the processed test features.
5. Save the predictions to a CSV file with the correct name.

In [ ]:
# Define path to the single participant data folder.
PARTICIPANT_DATA_PATH = '../participant_data/'
SUBMISSION_DIR = 'my_prediction_submission_cox'
os.makedirs(SUBMISSION_DIR, exist_ok=True)

# Define all 16 event pairs
index_events = ["Borrow", "Deposit", "Repay", "Withdraw"]
outcome_events = index_events + ["Liquidated"]
event_pairs = []
for index_event in index_events:
    for outcome_event in outcome_events:
        if index_event == outcome_event:
            continue
        event_pairs.append((index_event, outcome_event))

for index_event, outcome_event in event_pairs:
    print(f"\n{'='*50}")
    print(f"Processing and Predicting for: {index_event} -> {outcome_event}")
    print(f"{'='*50}")
    
    dataset_path = os.path.join(index_event, outcome_event)
    
    # --- Load and Preprocess ---
    try:
        train_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'train.csv'))
        # The test features for participants are the validation features.
        test_features_df = pd.read_csv(os.path.join(PARTICIPANT_DATA_PATH, dataset_path, 'test_features.csv'))
    except FileNotFoundError as e:
        print(f"Data not found for {dataset_path}. Skipping.")
        continue
        
    X_train, y_train, X_test_processed = preprocess(train_df, test_features_df)

    # --- Train Model ---
    try:
        lifelines_train_df = pd.concat([X_train, y_train.reset_index(drop=True)], axis=1)
        model = CoxPHFitter(penalizer=0.1)
        
        try:
            # First attempt to fit the model with standard parameters
            model.fit(lifelines_train_df, duration_col='timeDiff', event_col='status')
        except ConvergenceError:
            # If it fails, try again with a stronger penalizer and smaller step size
            print("  - Standard model failed to converge. Trying robust parameters...")
            model = CoxPHFitter(penalizer=1.0)
            model.fit(lifelines_train_df, duration_col='timeDiff', event_col='status', fit_options={'step_size': 0.1})
            print("  - Successfully converged with robust parameters.")

        # --- Generate and Save Predictions ---
        print(f"Generating predictions for {dataset_path}...")
        predictions = -model.predict_partial_hazard(X_test_processed)
        
        # Save predictions to a CSV file
        prediction_filename = dataset_path.replace(os.sep, '_') + '.csv'
        prediction_save_path = os.path.join(SUBMISSION_DIR, prediction_filename)
        pd.DataFrame(predictions).to_csv(prediction_save_path, header=False, index=False)
        print(f"  -> Predictions saved to {prediction_save_path}")
        
    except (ConvergenceError, ValueError) as e:
        print(f"\nERROR: The model for {dataset_path} failed to train even with robust parameters. No prediction file will be created.")
        print(f"Details: {e}")

print("\n\nAll prediction files have been generated.")


Processing and Predicting for: Borrow -> Deposit


## Step 4: Create the Submission ZIP file

The final step is to package all 16 of your prediction CSV files into a single `submission.zip` file.

In [ ]:
output_zip_filename = 'submission_cox'
shutil.make_archive(output_zip_filename, 'zip', SUBMISSION_DIR)

print(f"Successfully created '{output_zip_filename}.zip' from the '{SUBMISSION_DIR}' directory.")
print("You can now upload this file to the CodaBench competition.")